# Article 3 - Robust Smart Irrigation under Sensor Faults: Stress Testing and Fault-Aware Learning for IoT Greenhouses

**Reproduction notebook.** Standard vs fault-aware LightGBM under five sensor-fault families, sensor criticality and paired Wilcoxon tests (Tables I-III, Figs. 1-3, Sec. IV-E).

**Run time on Colab:** about 5-10 min.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/badrelkhalylyphd-boop/irrigation-sensor-fault-robustness/blob/main/reproduce_colab.ipynb)

### How to use
1. **Runtime > Run all.** The first cell installs the exact library versions used for the paper. On Colab the session then restarts by itself.
2. Upload `greenhouse_dataset.csv` in the **Files** panel of Colab (folder icon on the left; it goes to `/content`). The file is in `data/` of this repository or on [Mendeley Data](https://doi.org/10.17632/2tvpsht4s4.1). It survives the automatic restart of step 1.
3. **Runtime > Run all** a second time. Cell 2 reads `/content/greenhouse_dataset.csv` directly and checks its SHA-256.
4. The last section compares every number of the revised paper with this run and marks each one ✅ or ❌.

Locally (Jupyter), open the notebook from the repository root: the data file is picked up from `data/`.

## 1. Environment

In [ ]:
# Install the exact library versions used for the paper (runs once).
import os, sys, subprocess, importlib.metadata as md
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules
PINNED = {'numpy': '2.2.6', 'pandas': '2.3.3', 'scikit-learn': '1.7.2', 'scipy': '1.15.3', 'lightgbm': '4.7.0', 'matplotlib': '3.11.2', 'pyarrow': '20.0.0'}

def _version(pkg):
    try:
        return md.version(pkg)
    except md.PackageNotFoundError:
        return None

wrong = {p: _version(p) for p, v in PINNED.items() if _version(p) != v}
if wrong:
    print("Installing pinned versions (currently:", wrong, ") ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + [f"{p}=={v}" for p, v in PINNED.items()])
    if IN_COLAB:
        print("\nDone. The session restarts now. Then choose Runtime > Run all again.")
        os.kill(os.getpid(), 9)
    raise SystemExit("Pinned versions installed. Restart the kernel, then run all cells again.")
print("Environment OK -", sys.version.split()[0], PINNED)


## 2. Data

In [ ]:
# Get the dataset.
# On Colab: upload greenhouse_dataset.csv in the Files panel (left sidebar); it lands in /content and is read from there.
# Locally: the copy in data/ of the repository is used.
import hashlib, shutil, urllib.request
from pathlib import Path

COLAB_FILE = Path("/content/greenhouse_dataset.csv")   # file uploaded through the Colab Files panel
DATA_URL = ""  # optional, e.g. "https://raw.githubusercontent.com/badrelkhalylyphd-boop/irrigation-sensor-fault-robustness/main/data/greenhouse_dataset.csv"
EXPECTED_SHA256 = "e64a904a981d323f67210138cfc078e98d1911e6d2bb42195bf76aa818c51aff"

WORK = Path("work").resolve()
SRC, OUT = WORK / "src", WORK / "outputs"
SRC.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
DATA = WORK / "greenhouse_dataset.csv"

candidates = [COLAB_FILE, *sorted(Path("/content").glob("greenhouse_dataset*.csv")),
              Path("data/greenhouse_dataset.csv"), Path("../data/greenhouse_dataset.csv"), Path("greenhouse_dataset.csv")]
found = [p for p in candidates if p.is_file()]
if found:
    shutil.copy(found[0], DATA)
    print("Dataset read from", found[0])
elif DATA.exists():
    print("Dataset already in", DATA)
elif DATA_URL:
    urllib.request.urlretrieve(DATA_URL, DATA)
elif IN_COLAB:
    from google.colab import files
    print("greenhouse_dataset.csv not found in /content. Select it now (or upload it in the Files panel and rerun this cell).")
    name = next(iter(files.upload()))
    Path(name).replace(DATA)
else:
    raise FileNotFoundError("Put greenhouse_dataset.csv in data/ or next to this notebook, or set DATA_URL.")

sha = hashlib.sha256(DATA.read_bytes()).hexdigest()
print("SHA-256:", sha)
print("Same file as in the paper." if sha == EXPECTED_SHA256 else "WARNING: this is not the file used in the paper.")


## 3. Code
The cells below write the scripts of `src/` unchanged into `work/src/`.

In [ ]:
%%writefile work/src/prep3.py
"""Article 3 - preprocessing.
Usage: python prep3.py /path/to/greenhouse_dataset.csv
Output: irrigation_clean.parquet
"""
import sys
from datetime import datetime
import pandas as pd

FEATURES=['Soil Percent','Soil Raw','Suhu Tanah','Suhu Udara','Humidity']
path=sys.argv[1] if len(sys.argv)>1 else 'greenhouse_dataset.csv'
d=pd.read_csv(path,sep=';')
d=d[d['Label'].notna()].copy()
for c in FEATURES+['Label']:
    d[c]=pd.to_numeric(d[c],errors='coerce')

def parse_timestamp(s):
    for fmt in ('%Y-%m-%d_%H-%M-%S','%Y-%m-%d %H:%M:%S'):
        try: return datetime.strptime(str(s),fmt)
        except ValueError: pass
    return pd.NaT

d['ts']=d['Timestamp'].map(parse_timestamp)
d=d.dropna(subset=FEATURES+['Label','ts']).copy()
d['DateGroup']=d['ts'].dt.date.astype(str)
d=d.drop_duplicates(subset=FEATURES+['Label']).sort_values('ts').reset_index(drop=True)
d.to_parquet('irrigation_clean.parquet',index=False)
print('rows',len(d),'positives',int(d.Label.sum()),'days',d.DateGroup.nunique())


In [ ]:
%%writefile work/src/eval3.py
"""Article 3 - grouped sensor-fault stress test.
Usage: python eval3.py
Requires irrigation_clean.parquet from prep3.py.
Outputs article3_results.csv and article3_sensor_criticality.csv.
"""
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (f1_score,matthews_corrcoef,recall_score,precision_score,
    balanced_accuracy_score,average_precision_score,confusion_matrix)
from scipy.stats import wilcoxon
import lightgbm as lgb

SEED=42
FEATURES=['Soil Percent','Soil Raw','Suhu Tanah','Suhu Udara','Humidity']
d=pd.read_parquet('irrigation_clean.parquet')
X=d[FEATURES].astype(float); y=d.Label.astype(int).to_numpy(); groups=d.DateGroup.to_numpy()
cv=StratifiedGroupKFold(n_splits=5)  # no shuffle: deterministic, as used for the reported results
splits=list(cv.split(X,y,groups))

def build_model():
    return lgb.LGBMClassifier(n_estimators=150,num_leaves=31,learning_rate=.05,
        class_weight='balanced',random_state=SEED,n_jobs=-1,verbose=-1)

def augment(Xt,yt,std,rng):
    m=int(.5*len(Xt)); idx=rng.choice(len(Xt),size=m,replace=True)
    A=Xt.iloc[idx].copy().reset_index(drop=True); ya=yt[idx].copy(); z=A.to_numpy()
    blocks=np.array_split(np.arange(m),3)
    rr=blocks[0]; cc=rng.integers(0,A.shape[1],size=len(rr)); z[rr,cc]=np.nan
    rr=blocks[1]; cc=rng.integers(0,A.shape[1],size=len(rr)); z[rr,cc]+=rng.normal(0,.5,size=len(rr))*std[cc]
    rr=blocks[2]; cc=rng.integers(0,A.shape[1],size=len(rr)); z[rr,cc]+=rng.choice([-1,1],size=len(rr))*std[cc]
    A.iloc[:,:]=z
    return pd.concat([Xt.reset_index(drop=True),A],ignore_index=True), np.concatenate([yt,ya])

def corrupt(Xv,kind,std,med,rng):
    Z=Xv.copy(); a=Z.to_numpy(copy=True); n,p=a.shape
    if kind=='missing':
        mask=rng.random((n,p))<.08; a[mask]=np.nan
    elif kind=='noise':
        mask=rng.random((n,p))<.15; noise=rng.normal(size=a.shape)*std[None,:]*.5; a[mask]+=noise[mask]
    elif kind=='bias':
        j=int(rng.integers(0,p)); rows=rng.random(n)<.40; a[rows,j]+=rng.choice([-1,1])*std[j]
    elif kind=='stuck':
        j=int(rng.integers(0,p)); L=max(1,int(.25*n)); start=int(rng.integers(0,max(1,n-L))); a[start:start+L,j]=med[j]
    elif kind=='drift':
        j=int(rng.integers(0,p)); a[:,j]+=np.linspace(0,std[j],n)
    return pd.DataFrame(a,columns=Xv.columns,index=Xv.index)

def score(yt,pr,pb):
    tn,fp,fn,tp=confusion_matrix(yt,pr,labels=[0,1]).ravel()
    return dict(F1=f1_score(yt,pr),MCC=matthews_corrcoef(yt,pr),Recall=recall_score(yt,pr),
        Precision=precision_score(yt,pr,zero_division=0),BAcc=balanced_accuracy_score(yt,pr),
        PRAUC=average_precision_score(yt,pb),TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn))

models={'standard':[],'aware':[]}; stats=[]
for fi,(tr,va) in enumerate(splits):
    Xt=X.iloc[tr]; yt=y[tr]; std=Xt.std().replace(0,1).to_numpy(); med=Xt.median().to_numpy(); stats.append((std,med))
    models['standard'].append(build_model().fit(Xt,yt))
    Xa,ya=augment(Xt,yt,std,np.random.default_rng(500+fi))
    models['aware'].append(build_model().fit(Xa,ya))

FAULTS=['clean','missing','noise','bias','stuck','drift']; rows=[]
for seed in range(5):
    for training in ['standard','aware']:
        for fault in FAULTS:
            yy=[]; pp=[]; bb=[]
            for fi,(tr,va) in enumerate(splits):
                Xv=X.iloc[va]; yv=y[va]; std,med=stats[fi]
                Zv=Xv if fault=='clean' else corrupt(Xv,fault,std,med,np.random.default_rng(10000+seed*100+fi*10+FAULTS.index(fault)))
                pb=models[training][fi].predict_proba(Zv)[:,1]; pr=(pb>=.5).astype(int)
                yy.append(yv); pp.append(pr); bb.append(pb)
            rows.append(dict(seed=seed,training=training,fault=fault,**score(np.concatenate(yy),np.concatenate(pp),np.concatenate(bb))))

res=pd.DataFrame(rows); res.to_csv('article3_results.csv',index=False)

# Sensor-criticality experiment: complete single-channel faults.
crit=[]
for training in ['standard','aware']:
    for j,name in enumerate(FEATURES):
        for fault in ['missing','noise','bias','drift']:
            yy=[];pp=[];bb=[]
            for fi,(tr,va) in enumerate(splits):
                Xv=X.iloc[va].copy(); yv=y[va]; std,med=stats[fi]; a=Xv.to_numpy(copy=True); n=len(a)
                rng=np.random.default_rng(9000+j*100+fi*10+['missing','noise','bias','drift'].index(fault))
                if fault=='missing': a[:,j]=np.nan
                elif fault=='noise': a[:,j]+=rng.normal(0,std[j],size=n)
                elif fault=='bias': a[:,j]+=std[j]
                elif fault=='drift': a[:,j]+=np.linspace(0,std[j],n)
                Z=pd.DataFrame(a,columns=FEATURES,index=Xv.index); pb=models[training][fi].predict_proba(Z)[:,1]; pr=(pb>=.5).astype(int)
                yy.append(yv);pp.append(pr);bb.append(pb)
            crit.append(dict(training=training,sensor=name,fault=fault,**score(np.concatenate(yy),np.concatenate(pp),np.concatenate(bb))))
pd.DataFrame(crit).to_csv('article3_sensor_criticality.csv',index=False)

# Paired statistical tests over 25 fault realizations.
s=res[res.fault!='clean'].sort_values(['fault','seed'])
a=s[s.training=='aware']; b=s[s.training=='standard']
for metric in ['F1','MCC','PRAUC']:
    stat,p=wilcoxon(a[metric].to_numpy(),b[metric].to_numpy(),alternative='greater')
    print(metric,'mean gain',float((a[metric].to_numpy()-b[metric].to_numpy()).mean()),'p',p)
print(res.groupby(['training','fault'])[['F1','MCC','Recall','Precision','PRAUC']].agg(['mean','std']))


In [ ]:
%%writefile work/src/figs3.py
"""Generate the three article figures and compact table images."""
import pandas as pd, numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
OUT=Path('figures'); OUT.mkdir(exist_ok=True)
r=pd.read_csv('article3_results.csv'); c=pd.read_csv('article3_sensor_criticality.csv')
faults=['clean','missing','noise','bias','stuck','drift']
fig,ax=plt.subplots(figsize=(6.4,3.3),dpi=220); x=np.arange(len(faults)); w=.36
for i,tr in enumerate(['standard','aware']):
    g=r[r.training==tr].groupby('fault').F1.agg(['mean','std']).reindex(faults)
    ax.bar(x+(i-.5)*w,g['mean']*100,w,yerr=g['std']*100,capsize=2,label='Standard' if tr=='standard' else 'Fault-aware')
ax.set_xticks(x); ax.set_xticklabels(['Clean','Missing','Noise','Bias','Stuck','Drift']); ax.set_ylabel('F1-score (%)'); ax.set_ylim(0,60); ax.legend(frameon=False,ncol=2)
ax.grid(axis='y',alpha=.2); fig.tight_layout(); fig.savefig(OUT/'fig1_fault_f1.png',bbox_inches='tight'); plt.close(fig)
p=c[c.training=='standard'].pivot(index='sensor',columns='fault',values='F1').reindex(index=['Soil Percent','Soil Raw','Suhu Tanah','Suhu Udara','Humidity'],columns=['missing','noise','bias','drift'])
fig,ax=plt.subplots(figsize=(6.4,3.5),dpi=220); im=ax.imshow(p.values*100,aspect='auto',vmin=0,vmax=55); ax.set_xticks(range(4)); ax.set_xticklabels(['Missing','Noise','Bias','Drift'])
ax.set_yticks(range(5)); ax.set_yticklabels(['Soil moisture %','Soil raw','Soil temp.','Air temp.','Humidity'])
for i in range(5):
    for j in range(4): ax.text(j,i,f'{p.values[i,j]*100:.1f}',ha='center',va='center',fontsize=8)
fig.colorbar(im,ax=ax).set_label('F1-score (%)'); fig.tight_layout(); fig.savefig(OUT/'fig2_sensor_criticality.png',bbox_inches='tight'); plt.close(fig)
clean=r[r.fault=='clean'].groupby('training').F1.mean(); stress=r[r.fault!='clean'].groupby('training').F1.mean()
vals=[clean['standard']*100,stress['standard']*100,clean['aware']*100,stress['aware']*100]
fig,ax=plt.subplots(figsize=(5.8,3.1),dpi=220); ax.bar(['Std clean','Std faults','Aware clean','Aware faults'],vals)
for i,v in enumerate(vals): ax.text(i,v+1,f'{v:.1f}',ha='center',fontsize=8)
ax.set_ylabel('F1-score (%)'); ax.set_ylim(0,60); ax.grid(axis='y',alpha=.2); fig.tight_layout(); fig.savefig(OUT/'fig3_retention.png',bbox_inches='tight'); plt.close(fig)


In [ ]:
# Helper: run a script from work/src inside work/outputs and stream its output.
def run(script, *args):
    cmd = [sys.executable, "-u", str(SRC / script), *map(str, args)]
    env = dict(os.environ, MPLBACKEND="Agg", PYTHONIOENCODING="utf-8")
    p = subprocess.Popen(cmd, cwd=OUT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in p.stdout:
        if "Warning" not in line and "warnings.warn" not in line:
            print(line, end="")
    if p.wait():
        raise RuntimeError(f"{script} failed")


## 4. Run the experiment
All outputs go to `work/outputs/`.

### Step 1 - cleaning and day groups

In [ ]:
run("prep3.py", DATA)

### Step 2 - grouped CV, fault injection, criticality, Wilcoxon

In [ ]:
run("eval3.py")

### Step 3 - figures

In [ ]:
run("figs3.py")

## 5. Check against the revised paper

In [ ]:
# Compare every reported number with what this run produced.
import math
import pandas as pd
CHECKS = []

def chk(item, expected, obtained, tol=0.006):
    """Numbers are compared at the precision printed in the paper."""
    if isinstance(expected, str):
        ok = str(obtained) == expected
    else:
        ok = obtained is not None and not (isinstance(obtained, float) and math.isnan(obtained)) and abs(obtained - expected) <= tol
    CHECKS.append({"item": item, "paper": expected,
                   "this run": round(obtained, 4) if isinstance(obtained, float) else obtained,
                   "status": "✅" if ok else "❌"})

from scipy.stats import wilcoxon
clean = pd.read_parquet(OUT / "irrigation_clean.parquet")
chk("Sec. III-A | rows", 48642, len(clean), 0); chk("Sec. III-A | positives", 208, int(clean.Label.sum()), 0)
chk("Sec. III-A | days", 17, clean.DateGroup.nunique(), 0)
r = pd.read_csv(OUT / "article3_results.csv")
m = r.groupby(["training", "fault"])[["F1", "MCC", "Recall", "PRAUC"]].mean() * 100
for tr, vals in {"standard": (46.57, 53.14, 91.35, 76.21), "aware": (48.50, 54.27, 89.42, 71.80)}.items():
    for k, v in zip(["F1", "MCC", "Recall", "PRAUC"], vals):
        chk(f"Sec. IV-A | {tr} | clean {k}", v, m.loc[(tr, "clean"), k])
FAULT_F1 = {"standard": dict(missing=11.16, noise=40.78, bias=40.52, stuck=39.98, drift=48.08),
            "aware": dict(missing=44.26, noise=45.08, bias=44.32, stuck=41.37, drift=46.92)}
for tr, d in FAULT_F1.items():
    for fault, v in d.items():
        chk(f"Table I | {tr} | {fault} mean F1", v, m.loc[(tr, fault), "F1"])
stressed = r[r.fault != "clean"].groupby("training").F1.mean() * 100
chk("Sec. IV-A | standard | mean stressed F1", 36.11, stressed["standard"])
chk("Sec. IV-A | aware | mean stressed F1", 44.39, stressed["aware"])
chk("Sec. IV-A | standard | retention (%)", 77.5, 100 * stressed["standard"] / m.loc[("standard", "clean"), "F1"], 0.06)
chk("Sec. IV-A | aware | retention (%)", 91.5, 100 * stressed["aware"] / m.loc[("aware", "clean"), "F1"], 0.06)

c = pd.read_csv(OUT / "article3_sensor_criticality.csv").set_index(["training", "sensor", "fault"]).F1 * 100
for key, v in {("standard", "Soil Percent", "missing"): 1.3, ("standard", "Soil Percent", "noise"): 15.1,
               ("standard", "Suhu Udara", "bias"): 24.9, ("standard", "Suhu Udara", "missing"): 28.4,
               ("aware", "Soil Percent", "missing"): 21.9, ("aware", "Suhu Udara", "missing"): 60.8}.items():
    chk(f"Sec. IV-D | {' | '.join(key)} | F1", v, c.loc[key], 0.06)

s = r[r.fault != "clean"].sort_values(["fault", "seed"])
a, b = s[s.training == "aware"], s[s.training == "standard"]
for metric, p_exp in {"F1": 6.99e-5, "MCC": 1.39e-3}.items():
    p = wilcoxon(a[metric].to_numpy(), b[metric].to_numpy(), alternative="greater").pvalue
    chk(f"Sec. IV-E | Wilcoxon p ({metric}) x1e5", p_exp * 1e5, p * 1e5, 0.6 if metric == "F1" else 1)
gain = 100 * (a.F1.to_numpy() - b.F1.to_numpy()).mean()
chk("Sec. IV-E | mean F1 gain (points)", 8.28, gain)


In [ ]:
report = pd.DataFrame(CHECKS)
n_ok = (report.status == "✅").sum()
print(f"{n_ok} / {len(report)} values match the revised paper.")
pd.set_option("display.max_rows", 500)
report[report.status == "❌"] if n_ok < len(report) else print("All checks passed.")


Full list of checks:

In [ ]:
report

## 6. Figures

In [ ]:
# Figures produced by this run
from IPython.display import Image, display
for f in sorted(OUT.rglob("*.png")):
    print(f.name); display(Image(filename=str(f), width=520))
